# Setup

In [1]:
%env N_EX=64

env: N_EX=64


# Candidates

In [2]:
%%writefile get_candidates.py

import sys

sys.path.insert(0, "/kaggle/input/mdc-utils-v02/code")

import argparse
import os
import sys
from pathlib import Path

import pandas as pd
from omegaconf import OmegaConf
from utils.ingest_utils import _find_dataset_positions, check_presence, get_all_hits, load_data  # type: ignore
from vllm import LLM, SamplingParams

DOI_FIXER_PROMPT = "Extract the DOI from the provided text."


def sanitize_doi(row):
    org_doi = row["dataset_id"]
    doi_fixed = row["dataset_id_fixed"]
    if org_doi == doi_fixed:
        return org_doi

    pos = _find_dataset_positions(row["text"], doi_fixed.replace("https://doi.org/", ""))
    if len(pos) == 0:
        return org_doi

    if len(org_doi) >= len(doi_fixed):
        return org_doi
    if " " in doi_fixed:
        return org_doi
    return doi_fixed


def get_candidates(input_df, doi_bank, acc_bank, whitelist):
    doi_group_df = doi_bank.groupby("article_id")["dataset_id"].agg(list).reset_index()
    acc_group_df = acc_bank.groupby("article_id")["dataset_id"].agg(list).reset_index()

    dataset2family = dict(zip(acc_bank["dataset_id"], acc_bank["family"]))
    for doi in doi_bank["dataset_id"]:
        if doi not in dataset2family:
            dataset2family[doi] = "doi"

    article2doi = dict(zip(doi_group_df["article_id"], doi_group_df["dataset_id"]))
    article2acc = dict(zip(acc_group_df["article_id"], acc_group_df["dataset_id"]))

    hit_data = []
    for _, row in input_df.iterrows():
        article_id = row["article_id"]
        pdf_text = row["pdf_text"]
        xml_text = row["xml_text"]

        # try online lookup ---
        doi_hits = article2doi.get(article_id, [])
        acc_hits = article2acc.get(article_id.lower(), [])

        all_hits = list(set(doi_hits + acc_hits))

        selected_hits = []
        for hit in all_hits:
            if check_presence(pdf_text, hit) or check_presence(xml_text, hit):
                selected_hits.append(hit)

        if len(selected_hits) > 0:
            hit_data.extend([{"article_id": article_id, "dataset_id": hit} for hit in selected_hits])
            continue

        # look for match using regex
        pdf_hits = get_all_hits(pdf_text, whitelist=whitelist)
        xml_hits = get_all_hits(xml_text, whitelist=whitelist)
        all_hits = list(set(pdf_hits + xml_hits))
        hit_data.extend([{"article_id": article_id, "dataset_id": hit} for hit in all_hits])

    candidate_df = pd.DataFrame(hit_data)
    candidate_df["family"] = candidate_df["dataset_id"].map(dataset2family)
    candidate_df["family"] = candidate_df["family"].fillna("custom")
    candidate_df["family"] = candidate_df.apply(lambda row: "doi" if row["dataset_id"].startswith("https://doi.org") else row["family"], axis=1)
    return candidate_df


def fix_multiline_doi(cfg, candidate_df, input_df, offset=32):
    original_df = candidate_df.copy()
    focus_df = candidate_df[(candidate_df["family"] == "custom") & (candidate_df["dataset_id"].str.startswith("https://doi.org"))].reset_index(drop=True)
    print(f"# of DOI with custom family: {len(focus_df)}")

    if len(focus_df) == 0:
        return original_df

    # prepare data for doi fixer
    infer_data = []
    for _, row in focus_df.iterrows():
        article_id = row["article_id"]
        dataset_id = row["dataset_id"]
        text_row = input_df[input_df["article_id"] == article_id]

        text_pdf = text_row["pdf_text"].values[0]
        text_xml = text_row["xml_text"].values[0]
        text = text_pdf + text_xml

        search_id = dataset_id.replace("https://doi.org/", "")
        positions = _find_dataset_positions(text, search_id)[:2]  # take up to 2 positions
        for start, end in positions:
            window = text[start : end + offset]
            infer_data.append({"article_id": article_id, "dataset_id": dataset_id, "text": window})
    infer_df = pd.DataFrame(infer_data)

    # DOI Fixer LLM ---
    llm = LLM(
        model=cfg.doi_fixer_model,
        dtype="half",
        tensor_parallel_size=2,
        max_model_len=512,
        enforce_eager=True,
        gpu_memory_utilization=0.9,
        enable_prefix_caching=False,
        disable_log_stats=True,
        max_num_seqs=64,
        swap_space=0,
        cpu_offload_gb=0,
        max_num_batched_tokens=2048,
    )

    # create prompts ---
    prompts = []
    for _, row in infer_df.iterrows():
        user_message = f"# Text:\n\n{row['text']}"
        prompt = f"{DOI_FIXER_PROMPT}\n\n{user_message}\n\nAnswer:\n"
        prompts.append(prompt)

    # print a few prompts
    for p in prompts[:1]:
        print(p)
        print("-" * 100)

    batch_size = 4096
    extracted_doi_list = []

    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i : i + batch_size]
        print(f"Processing batch {i // batch_size + 1}/{(len(prompts) + batch_size - 1) // batch_size} (items {i + 1}-{min(i + batch_size, len(prompts))})")
        sampling_params = SamplingParams(temperature=0.0, skip_special_tokens=True, max_tokens=128)
        batch_outputs = llm.generate(batch_prompts, sampling_params=sampling_params, use_tqdm=True)

        for output in batch_outputs:
            extracted_doi_list.append(output.outputs[0].text)

    # fixed
    fixed_df = pd.DataFrame()
    fixed_df["article_id"] = infer_df["article_id"]
    fixed_df["dataset_id"] = infer_df["dataset_id"]
    fixed_df["family"] = "custom"
    fixed_df["text"] = infer_df["text"]
    fixed_df["dataset_id_fixed"] = extracted_doi_list
    fixed_df["dataset_id_fixed"] = fixed_df["dataset_id_fixed"].apply(lambda x: f"https://doi.org/{x}")
    fixed_df["dataset_id_fixed"] = fixed_df.apply(sanitize_doi, axis=1)

    fixed_df = fixed_df[["article_id", "dataset_id_fixed", "family"]].rename(columns={"dataset_id_fixed": "dataset_id"})

    candidate_df = pd.concat([fixed_df, original_df]).reset_index(drop=True)
    candidate_df = candidate_df.drop_duplicates(subset=["article_id", "dataset_id"], keep="first").reset_index(drop=True)

    return candidate_df


def main(cfg):
    if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        data_dir = Path("/kaggle/input/make-data-count-finding-data-references/test")
    else:
        data_dir = Path("/kaggle/input/make-data-count-finding-data-references/train")

    cache_dir = Path("cache")
    cache_dir.mkdir(parents=True, exist_ok=True)

    input_df = load_data(data_dir, cache_dir)

    if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        input_df = input_df.sort_values(by="article_id").reset_index(drop=True)
        n_ex = int(os.getenv("N_EX"))
        input_df = input_df.head(n_ex)

    current_ids = set(input_df["article_id"].tolist())
    current_ids_lower = set(x.lower() for x in current_ids)

    print(f"# of articles: {len(current_ids)}")

    # helper data ---
    whitelist_df = pd.read_parquet(cfg.whitelist_path)
    whitelist = whitelist_df["prefix"].tolist()

    doi_index_df = pd.read_parquet(cfg.doi_lookup_path)
    acc_index_df = pd.read_parquet(cfg.acc_lookup_path)

    doi_bank = doi_index_df[doi_index_df["article_id"].isin(current_ids)].reset_index(drop=True)
    acc_bank = acc_index_df[acc_index_df["article_id"].isin(current_ids_lower)].reset_index(drop=True)

    print("--" * 50)
    print(f"# of articles DOI found: {doi_bank['article_id'].nunique()} / {len(current_ids)}")
    print(f"# of articles ACC found: {acc_bank['article_id'].nunique()} / {len(current_ids)}")

    candidate_df = get_candidates(input_df, doi_bank, acc_bank, whitelist)
    candidate_df = candidate_df[candidate_df["family"] != "rrid"].reset_index(drop=True)  # remove rrid

    print("--" * 50)
    print(f"# of candidate documents: {len(candidate_df)}")
    print(candidate_df["family"].value_counts())
    print("--" * 50)

    print("Fixing multiline DOI...")
    print(f"shape before fixing: {candidate_df.shape}")
    candidate_df = fix_multiline_doi(cfg, candidate_df, input_df)
    print(f"shape after fixing: {candidate_df.shape}")

    # Save
    save_dir = cfg.save_dir
    save_path = os.path.join(save_dir, "candidates.parquet")
    candidate_df.to_parquet(save_path, index=False)
    print(f"saved online citations to: {save_path}")

    index_df = pd.concat([doi_bank[["article_id", "dataset_id"]], acc_bank[["article_id", "dataset_id"]]]).reset_index(drop=True)
    save_path = os.path.join(save_dir, "index.parquet")
    index_df.to_parquet(save_path, index=False)
    print("--" * 50)

    save_path = os.path.join(save_dir, "inputs.parquet")
    input_df.to_parquet(save_path, index=False)

    # save doi lookup for later use
    save_path = os.path.join(save_dir, "doi_bank.parquet")
    doi_bank.to_parquet(save_path, index=False)


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config_path", type=str, default="conf/eval.yaml")
    args = parser.parse_args()

    cfg = OmegaConf.load(args.config_path)

    os.makedirs(cfg.save_dir, exist_ok=True)
    main(cfg)

Writing get_candidates.py


In [3]:
%%writefile config_candidates.yaml

save_dir: ./working
whitelist_path: /kaggle/input/mdc-data-doi-prefix/doi_prefix_whitelist.parquet
doi_lookup_path: /kaggle/input/mdc-corpus-processed-v3/index.parquet
acc_lookup_path: /kaggle/input/mdc-accession-bank/accession_bank.parquet
doi_fixer_model: /kaggle/input/mdc_doi_fixer/transformers/default/2

Writing config_candidates.yaml


In [4]:
%%time
!python get_candidates.py --config_path config_candidates.yaml

INFO 09-08 22:44:42 [__init__.py:244] Automatically detected platform cuda.
2025-09-08 22:44:44.247832: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757371484.437838      72 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757371484.490635      72 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Cache directory (cache/exp_tmp/txt) exists? -> False
MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: unsupported error: cannot create appearance stream for  annotations

MuPDF error: unsupported error: cannot create appearance strea

In [5]:
# import pandas as pd
# df = pd.read_parquet("./working/candidates.parquet")
# df # [article_id, dataset_id, familty]

# Filter

In [6]:
%%writefile filter_candidates.py

import sys

sys.path.insert(0, "/kaggle/input/mdc-utils-v02/code")

import argparse
import json
import os
import random
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from omegaconf import OmegaConf
from transformers import AutoTokenizer
from utils.ingest_utils import create_chunks, gt_dataset_id_normalization, load_data  # type: ignore
from vllm import LLM, SamplingParams  # type: ignore

SYS_PROMPT = """You are an expert at identifying research data citations in scientific literature. You excel at identifying and cataloging data citations in academic papers.

Your task is to determine whether a given DOI or Accession ID represents a valid citation based on the provided text context. Focus on identifying genuine data citations, not just any mention of an identifier."""

USER_TEMPLATE_W_DOI = """Text snippet: {context}

Detected DOI in the whole article: {detected_dois}

Focus DOI/Accession ID: {dataset_id}

Is {dataset_id} a valid research data citation in this text snippet?

Respond with only Yes or No"""

USER_TEMPLATE_WO_DOI = """Text snippet: {context}

Focus DOI/Accession ID: {dataset_id}

Is {dataset_id} a valid research data citation in this text snippet?

Respond with only Yes or No"""


def get_yes_prob(logprobs):
    yes_logit = None
    no_logit = None

    for _, logprob_obj in logprobs.items():
        token_str = logprob_obj.decoded_token.strip().lower()

        if token_str == "yes":
            if yes_logit is None:
                yes_logit = logprob_obj.logprob
        elif token_str == "no":
            if no_logit is None:
                no_logit = logprob_obj.logprob

    default_val = -20.0
    if yes_logit is None:
        yes_logit = default_val

    if no_logit is None:
        no_logit = default_val

    logits = np.array([yes_logit, no_logit])
    logits_max = np.max(logits)
    exp_logits = np.exp(logits - logits_max)
    normalized_scores = exp_logits / np.sum(exp_logits)

    return normalized_scores[0]


def get_chunk_hits(row, article2hits):
    aid = row["article_id"]
    chunk = row["chunk"]
    chunk = re.sub(r"\s+", "", chunk).lower()

    labels = article2hits.get(aid, [])

    ret = []
    for label in labels:
        search_id = gt_dataset_id_normalization(label)
        if search_id in chunk:
            ret.append(label)

    return ret


def get_doi_mapping(index_df):
    gdf = index_df.groupby("article_id")["dataset_id"].agg(set).reset_index()
    article2dataset_online = dict(zip(gdf["article_id"], gdf["dataset_id"]))

    article2doi_online = {k: [y for y in v if y.startswith("https://doi.org")] for k, v in article2dataset_online.items()}
    article2doi_online = {k: [y.replace("https://doi.org/", "") for y in v] for k, v in article2doi_online.items()}
    article2doi_online = {k: " | ".join(v) for k, v in article2doi_online.items()}

    return article2doi_online


def filter_fn(cfg, row):
    doi_th = cfg.doi_th

    acc_th = cfg.acc_th

    is_doi = row["dataset_id"].startswith("https://doi.org/")
    if is_doi:
        return row["yes_prob"] >= doi_th
    else:
        return row["yes_prob"] >= acc_th


def main(cfg):
    if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        data_dir = Path("/kaggle/input/make-data-count-finding-data-references/test")
    else:
        data_dir = Path("/kaggle/input/make-data-count-finding-data-references/train")

    cache_dir = Path("cache")
    cache_dir.mkdir(parents=True, exist_ok=True)

    input_df = load_data(data_dir, cache_dir)

    if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        input_df = input_df.sort_values(by="article_id").reset_index(drop=True)
        n_ex = int(os.getenv("N_EX"))
        input_df = input_df.head(n_ex)

    # separate into custom vs non-custom
    hit_df = pd.read_parquet(cfg.extraction_path)[["article_id", "dataset_id", "family"]]
    custom_article_ids = set(hit_df[hit_df["family"] == "custom"]["article_id"].unique().tolist())

    if len(custom_article_ids) == 0:
        print("No custom articles found, skipping filter")
        hit_df.to_parquet(os.path.join(cfg.save_dir, "filtered_candidates.parquet"), index=False)
        return  # no need to filter non-custom

    input_df = input_df[input_df["article_id"].isin(custom_article_ids)].reset_index(drop=True)
    hit_df_custom = hit_df[hit_df["family"] == "custom"].reset_index(drop=True)
    hit_df_non_custom = hit_df[hit_df["family"] != "custom"].reset_index(drop=True)

    print(f"shape of hit_df_custom: {hit_df_custom.shape}")
    print(f"shape of hit_df_non_custom: {hit_df_non_custom.shape}")

    chunk_tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
    chunk_df = create_chunks(input_df, chunk_tokenizer, tokens_per_chunk=cfg.tokens_per_chunk, chunk_overlap=cfg.chunk_overlap)

    if cfg.xml_first:
        chunk_df = chunk_df.sort_values(by="source", ascending=False).reset_index(drop=True)  # Use XML if available

    # load candidate data
    gdf = hit_df_custom.groupby("article_id")["dataset_id"].agg(list).reset_index()
    article2hits = dict(zip(gdf["article_id"], gdf["dataset_id"]))

    chunk_df["candidate_ids"] = chunk_df.apply(lambda row: get_chunk_hits(row, article2hits), axis=1)
    candidate_df = chunk_df.explode("candidate_ids").dropna(subset=["candidate_ids"]).rename(columns={"candidate_ids": "dataset_id", "chunk": "context"})
    candidate_df = candidate_df.drop_duplicates(subset=["article_id", "dataset_id"], keep="first").reset_index(drop=True)
    candidate_df = candidate_df.sort_values(by="context").reset_index(drop=True)

    candidate_df = candidate_df[["article_id", "dataset_id", "context", "source"]]
    samp = candidate_df.sample().to_dict(orient="records")[0]
    print(json.dumps(samp))

    ds = Dataset.from_pandas(candidate_df)
    print(f"Number of examples: {len(ds)}")

    max_model_len = cfg.max_model_len

    llm = LLM(
        model=cfg.model_name,
        dtype="half",
        tensor_parallel_size=2,
        max_model_len=max_model_len,
        enforce_eager=True,
        gpu_memory_utilization=0.9,
        enable_prefix_caching=True,
        disable_log_stats=True,
        max_num_seqs=16,
        swap_space=0,
        cpu_offload_gb=0,
        max_num_batched_tokens=4096,
    )

    tokenizer = llm.get_tokenizer()

    index_df = pd.read_parquet(cfg.lookup_path)
    article2doi_online = get_doi_mapping(index_df)

    print("MODEL IS LOADED")

    # create prompts ---
    prompts = []
    print("CREATING PROMPTS...")
    for i, row in candidate_df.iterrows():
        article_id = row["article_id"]
        dataset_id = row["dataset_id"]
        context = row["context"]

        detected_dois = article2doi_online.get(article_id, "")  # Missing
        if len(detected_dois.strip()) == 0:
            detected_dois = "N/A"

        if (article_id in article2doi_online) and (cfg.use_online_doi):
            USER_TEMPLATE = USER_TEMPLATE_W_DOI
        else:
            USER_TEMPLATE = USER_TEMPLATE_WO_DOI

        user_prompt = USER_TEMPLATE.format(context=context, dataset_id=dataset_id, detected_dois=detected_dois)
        messages = [{"role": "system", "content": SYS_PROMPT}, {"role": "user", "content": user_prompt}]

        prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

        # max len
        tokens = tokenizer.encode(prompt)
        token_length = len(tokens)
        extra_tokens = token_length - (max_model_len - 16)

        if extra_tokens > 0:
            print(f"Prompt length: {token_length} > {max_model_len} - 16")
            print(f"Prompt: {prompt}")
            print("-" * 100)
            prompt = tokenizer.decode(tokens[extra_tokens:])

        prompts.append(prompt)

    print("DONE CREATING PROMPTS")

    # print a few prompts
    if len(prompts) >=2:
        samples = random.sample(prompts, 2)
        for p in samples:
            print(p)
            print("-" * 100)

    print("RUNNING OUTPUT GEN")
    batch_size = 4096
    probs_list = []

    # Process in batches
    for i in range(0, len(prompts), batch_size):
        batch_prompts = prompts[i : i + batch_size]
        print(f"Processing batch {i // batch_size + 1}/{(len(prompts) + batch_size - 1) // batch_size} (items {i + 1}-{min(i + batch_size, len(prompts))})")

        # Generate outputs for this batch
        batch_outputs = llm.generate(batch_prompts, SamplingParams(temperature=0.0, skip_special_tokens=True, max_tokens=1, logprobs=20), use_tqdm=True)

        # Extract probabilities for this batch
        for output in batch_outputs:
            lps = output.outputs[0].logprobs[0]
            probs_list.append(get_yes_prob(lps))

    sub_df = pd.DataFrame()
    sub_df["article_id"] = candidate_df["article_id"]
    sub_df["dataset_id"] = candidate_df["dataset_id"]
    sub_df["yes_prob"] = probs_list
    sub_df = sub_df.sort_values(by="yes_prob", ascending=False).reset_index(drop=True)
    sub_df = sub_df.drop_duplicates(subset=["article_id", "dataset_id"], keep="first").reset_index(drop=True)

    oof_df = sub_df.copy()
    oof_df["should_keep"] = oof_df.apply(lambda x: filter_fn(cfg, x), axis=1)
    oof_df = oof_df[oof_df["should_keep"]].reset_index(drop=True)
    oof_df = oof_df.drop(columns=["should_keep"])
    oof_df = oof_df.drop_duplicates(subset=["article_id", "dataset_id"], keep="first").reset_index(drop=True)
    oof_df["family"] = "custom"
    oof_df = oof_df[["article_id", "dataset_id", "family"]].copy()

    # merge with non-custom
    print(f"shape before filter: {hit_df.shape}")
    oof_df = pd.concat([oof_df, hit_df_non_custom[["article_id", "dataset_id", "family"]]], ignore_index=True)
    oof_df = oof_df.drop_duplicates(subset=["article_id", "dataset_id"], keep="first").reset_index(drop=True)

    save_dir = cfg.save_dir
    os.makedirs(save_dir, exist_ok=True)

    oof_df.to_parquet(os.path.join(save_dir, "filtered_candidates.parquet"), index=False)
    print(f"shape after filter: {oof_df.shape}")


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config_path", type=str, default="conf.yaml")
    args = parser.parse_args()

    cfg = OmegaConf.load(args.config_path)
    main(cfg)

    if torch.distributed.is_initialized():
        torch.distributed.destroy_process_group()

Writing filter_candidates.py


In [7]:
%%writefile config_filter.yaml

save_dir: ./working
extraction_path: ./working/candidates.parquet

doi_th: 0.3
acc_th: 0.1

tokens_per_chunk: 1296
chunk_overlap: 256

model_name: /kaggle/input/mdc_filter_qwen2b/transformers/default/1
max_model_len: 2048

xml_first: false
use_online_doi: true

lookup_path: ./working/doi_bank.parquet

Writing config_filter.yaml


In [8]:
%%time
!python filter_candidates.py --config_path config_filter.yaml

INFO 09-08 22:46:51 [__init__.py:244] Automatically detected platform cuda.
2025-09-08 22:46:51.852793: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1757371611.876643      92 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1757371611.884074      92 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Cache directory (cache/exp_tmp/txt) exists? -> True
# of previously processed files: 524
# of new files processed: 0
shape of hit_df_custom: (2, 3)
shape of hit_df_non_custom: (101, 3)
Processing 2 articles using 8 jobs...
Preparing parallel jobs: 100%|███████████████████| 2/2 [00:00<00:00, 333.77it/s]
[Parallel(n_jobs=8)]: Done   2 out of   2 | elapsed:    4

# Prepare for type prediction

In [9]:
%%writefile prep_type_inputs.py

import argparse
import sys

import pandas as pd
from omegaconf import OmegaConf

sys.path.insert(0, "/kaggle/input/mdc-utils-v02/code")
from utils.ingest_utils import check_presence, score_context, split_text  # type: ignore


def main(cfg):
    candidate_df = pd.read_parquet(cfg.extraction_path)
    input_df = pd.read_parquet(cfg.input_path)

    # Split Candidates --
    candidate_df_doi = candidate_df[candidate_df["dataset_id"].str.startswith("https://doi.org")].reset_index(drop=True)
    candidate_df_acc = candidate_df[~candidate_df["dataset_id"].str.startswith("https://doi.org")].reset_index(drop=True)

    articles_with_doi = set(candidate_df_doi["article_id"].tolist())
    remove_acc_articles = set()

    for aid in articles_with_doi:
        input_row = input_df[input_df["article_id"] == aid]
        pdf_text = input_row["pdf_text"].values[0]
        xml_text = input_row["xml_text"].values[0]
        pdf_chunks = split_text(pdf_text)
        xml_chunks = split_text(xml_text)
        all_chunks = pdf_chunks + xml_chunks

        # check chunks with doi
        curr_doi_list = candidate_df_doi[candidate_df_doi["article_id"] == aid]["dataset_id"].tolist()

        hit_chunks = []
        for chunk in all_chunks:
            if any([check_presence(chunk, d) for d in curr_doi_list]):
                hit_chunks.append(chunk)

        # check hit scores
        for chunk in hit_chunks:
            if score_context(chunk) >= 5.0:
                remove_acc_articles.add(aid)
                break

    # remove accessions
    print(f"shape of candidate_df_acc before removing accessions: {candidate_df_acc.shape}")
    candidate_df_acc["should_remove"] = candidate_df_acc["article_id"].apply(lambda x: x in remove_acc_articles)
    candidate_df_acc = candidate_df_acc[~candidate_df_acc["should_remove"]].reset_index(drop=True)
    candidate_df_acc = candidate_df_acc.drop(columns=["should_remove"])
    print(f"shape of candidate_df_acc after removing accessions: {candidate_df_acc.shape}")

    # candidates that needs type predictions
    base_df = pd.concat([candidate_df_doi, candidate_df_acc], ignore_index=True)
    base_df = base_df.drop_duplicates(subset=["article_id", "dataset_id"], keep="first").reset_index(drop=True)
    base_df.to_parquet("./working/base_pred_df.parquet")

    # assume type
    assume_secondary_list = set(["cath", "alphafold", "cellosaurus", "chembl", "dbgap", "igsr", "pfam", "reactome", "refseq"])
    candidate_df_acc_secondary = candidate_df_acc[candidate_df_acc["family"].isin(assume_secondary_list)].reset_index()
    candidate_df_acc_secondary["probs"] = [[0.0, 1.0, 0.0] for _ in range(len(candidate_df_acc_secondary))]
    candidate_df_acc_secondary["type"] = "Secondary"
    candidate_df_acc_secondary = candidate_df_acc_secondary[["article_id", "dataset_id", "probs", "type"]]
    candidate_df_acc_secondary.to_parquet("./working/acc_preds_assumed.parquet")

    # infer
    candidate_df_acc_infer = candidate_df_acc[~candidate_df_acc["family"].isin(assume_secondary_list)].reset_index(drop=True)
    print(f"shape of candidate_df_acc_infer before clipping: {candidate_df_acc_infer.shape}")

    # candidate_df_acc_infer = candidate_df_acc_infer.drop_duplicates(subset=["article_id", "dataset_id"], keep="first").groupby(["article_id", "family"], sort=False).head(cfg.max_acc_per_family).reset_index(drop=True)
    # candidate_df_acc_infer = candidate_df_acc_infer.drop_duplicates(subset=["article_id", "dataset_id"], keep="first").groupby(["article_id", "family"], sort=False, group_keys=False).apply(lambda grp: grp.sample(n=min(cfg.max_acc_per_family, len(grp)), random_state=42)).reset_index(drop=True)
    candidate_df_acc_infer = candidate_df_acc_infer.drop_duplicates(subset=["article_id", "dataset_id"], keep="first").sample(frac=1, random_state=42).groupby(["article_id", "family"], sort=False).head(cfg.max_acc_per_family).reset_index(drop=True)
    print(f"shape of candidate_df_acc_infer after clipping: {candidate_df_acc_infer.shape}")


    infer_df = pd.concat([candidate_df_acc_infer, candidate_df_doi], ignore_index=True)
    infer_df = infer_df.sort_values(by="article_id").reset_index(drop=True)
    infer_df.to_parquet("./working/candidates_infer.parquet")

    #---
    print(f"# of rows for inference: {len(infer_df)}")
    print(f"# of candidates: {len(base_df)}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--config_path", type=str, default="conf/pipeline.yaml")
    args = ap.parse_args()
    cfg = OmegaConf.load(args.config_path)
    main(cfg)

Writing prep_type_inputs.py


In [10]:
%%writefile config_prep.yaml

save_dir: ./working
extraction_path: ./working/filtered_candidates.parquet
input_path: ./working/inputs.parquet
max_acc_per_family: 8

Writing config_prep.yaml


In [11]:
!python prep_type_inputs.py --config_path config_prep.yaml

shape of candidate_df_acc before removing accessions: (48, 3)
shape of candidate_df_acc after removing accessions: (11, 3)
shape of candidate_df_acc_infer before clipping: (1, 3)
shape of candidate_df_acc_infer after clipping: (1, 3)
# of rows for inference: 54
# of candidates: 64


In [12]:
# infer_df = pd.read_parquet("./working/base_pred_df.parquet")
# candidate_df = pd.read_parquet("./working/candidates_infer.parquet")

# # # Split Candidates ---
# candidate_df_doi = candidate_df[candidate_df["dataset_id"].str.startswith("https://doi.org")].reset_index(drop=True)
# candidate_df_acc = candidate_df[~candidate_df["dataset_id"].str.startswith("https://doi.org")].reset_index(drop=True)

In [13]:
# infer_df.shape, candidate_df.shape, candidate_df_doi.shape, candidate_df_acc.shape

In [14]:
# candidate_df_doi

In [15]:
# candidate_df_acc = candidate_df_acc.drop_duplicates(subset=['article_id', 'dataset_id'])
# candidate_df_acc.shape

In [16]:
# candidate_df_doi = candidate_df_doi.drop_duplicates(subset=['article_id', 'dataset_id'])
# candidate_df_doi.shape

# Type Classification

In [17]:
%%writefile classify_candidates.py

import sys

sys.path.insert(0, "/kaggle/input/mdc-utils-v02/code")

import argparse
import os
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import vllm
from omegaconf import OmegaConf
from pandarallel import pandarallel
from transformers import AutoTokenizer

code_dir = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, code_dir)

from utils.ingest_utils import _find_dataset_positions, check_presence, load_data, get_context, get_context_v2, get_context_many_accessions

SYS_PROMPT = """You are an expert at analyzing research data usage in academic papers. You will be shown a snippet of text likely containing one or more Accession IDs and/or DOIs.

Analyze the context to determine how a given Accession ID and/or DOI is being used:

A) PRIMARY - The authors generated this data for their current study
B) SECONDARY - The authors are using existing data from other sources
C) NONE - This is not a research data citation"""

USER_TEMPLATE = """# Context:\n\n{context}

# Focus DOI/Accession ID: **{dataset_id}**

# Task: Classify {dataset_id} as:
A) PRIMARY - Dataset created by these authors for this paper
B) SECONDARY - Dataset from previous work being reused
C) NONE - Not a dataset citation

Respond with only one letter: A, B, or C."""

os.environ["TOKENIZERS_PARALLELISM"] = "false"
pandarallel.initialize(progress_bar=True, nb_workers=4)


def get_probs(logprobs):
    a_logit = None
    b_logit = None
    c_logit = None

    for _, logprob_obj in logprobs.items():
        token_str = logprob_obj.decoded_token.strip().lower()

        if token_str == "a":
            if a_logit is None:
                a_logit = logprob_obj.logprob
        elif token_str == "b":
            if b_logit is None:
                b_logit = logprob_obj.logprob
        elif token_str == "c":
            if c_logit is None:
                c_logit = logprob_obj.logprob

    default_val = -20.0
    if a_logit is None:
        a_logit = default_val

    if b_logit is None:
        b_logit = default_val

    if c_logit is None:
        c_logit = default_val

    logits = np.array([a_logit, b_logit, c_logit])
    logits_max = np.max(logits)
    exp_logits = np.exp(logits - logits_max)
    normalized_scores = exp_logits / np.sum(exp_logits)

    return normalized_scores


def get_doi_mapping(index_df, input_df):
    gdf = index_df.groupby("article_id")["dataset_id"].agg(set).reset_index()
    article2dataset_online = dict(zip(gdf["article_id"], gdf["dataset_id"]))

    article_to_row = {row["article_id"]: row for _, row in input_df.iterrows()}

    article2doi_online = {}
    for article_id, dataset_ids in article2dataset_online.items():
        if article_id not in article_to_row:
            continue

        doi_candidates = [y for y in dataset_ids if y.startswith("https://doi.org")]

        row = article_to_row[article_id]
        filtered_dois = []

        for doi in doi_candidates:
            if check_presence(row["pdf_text"], doi) or check_presence(row["xml_text"], doi):
                filtered_dois.append(doi)
            article2doi_online[article_id] = filtered_dois

    return article2doi_online


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--config_path", type=str, default="conf/eval.yaml")
    args = parser.parse_args()

    cfg = OmegaConf.load(args.config_path)

    if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        data_dir = Path("/kaggle/input/make-data-count-finding-data-references/test")
    else:
        data_dir = Path("/kaggle/input/make-data-count-finding-data-references/train")

    # ---
    cache_dir = Path("cache")
    cache_dir.mkdir(parents=True, exist_ok=True)

    input_df = load_data(data_dir, cache_dir)

    if not os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
        input_df = input_df.sort_values(by="article_id").reset_index(drop=True)
        n_ex = int(os.getenv("N_EX"))
        input_df = input_df.head(n_ex)

    current_ids = set(input_df["article_id"].tolist())
    current_ids_lower = set(x.lower() for x in current_ids)
    chunk_tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

    # ---------
    candidate_df = pd.read_parquet(cfg.candidate_path) # article_id, dataset_id
    candidate_df = candidate_df.drop_duplicates(subset=["article_id", "dataset_id"], keep="first").reset_index(drop=True)
    candidate_df = candidate_df[["article_id", "dataset_id"]]
    print(f"shape of candidates: {candidate_df.shape}")

    # add the text fields ---
    data = []
    for _, row in input_df.iterrows():
        article_id = row["article_id"]
        pdf_text = row["pdf_text"]
        xml_text = row["xml_text"]

        data.append({"article_id": article_id, "text": pdf_text, "source": "pdf"})
        data.append({"article_id": article_id, "text": xml_text, "source": "xml"})

    text_df = pd.DataFrame(data)

    candidate_df = candidate_df.merge(text_df, on=["article_id"])
    candidate_df = candidate_df[["article_id", "dataset_id", "text", "source"]]

    # filter out rows for which dataset_id is not in the text
    print(f"shape before search for dataset_id in text: {candidate_df.shape}")
    candidate_df = candidate_df[candidate_df.apply(lambda row: check_presence(row["text"], row["dataset_id"]), axis=1)].reset_index(drop=True)
    print(f"shape after search for dataset_id in text: {candidate_df.shape}")
    if cfg.xml_first:
        candidate_df = candidate_df.sort_values(by="source", ascending=False).reset_index(drop=True)  # Use XML if available
            
    candidate_df = candidate_df.drop_duplicates(subset=["article_id", "dataset_id"], keep="first").reset_index(drop=True)
    print(f"shape after deduplication: {candidate_df.shape}")

    index_df = pd.read_parquet("./working/index.parquet")
    doi_map = get_doi_mapping(index_df, input_df)
    candidate_df["detected_dois"] = candidate_df["article_id"].map(lambda x: doi_map.get(x, []))

    ##########################################
    candidate_df_doi = candidate_df[candidate_df["dataset_id"].str.startswith("https://doi.org")].reset_index(drop=True)
    candidate_df_acc = candidate_df[~candidate_df["dataset_id"].str.startswith("https://doi.org")].reset_index(drop=True)
    
    base_df = pd.read_parquet("./working/base_pred_df.parquet") # article_id, dataset_id, family
    candidate_df_acc = candidate_df_acc.merge(base_df, on=['article_id', 'dataset_id'], how='left')
    candidate_df_acc['family'] = candidate_df_acc['family'].fillna('custom')
    
    count_df = candidate_df_acc.groupby(["article_id", "family"])["dataset_id"].agg(lambda x: len(set(x))).reset_index().rename(columns={"dataset_id": "count"})
    gdf = candidate_df_acc.groupby(["article_id", "family"])["dataset_id"].agg(list).reset_index().rename(columns={"dataset_id": "detected_accession_ids"})
    long_table_articles = set(count_df[count_df["count"] >= cfg.max_accession_ids_per_family]["article_id"].tolist())
    candidate_df_acc_long = candidate_df_acc[candidate_df_acc["article_id"].isin(long_table_articles)].reset_index(drop=True)
    candidate_df_acc_long_infer = candidate_df_acc_long
    candidate_df_acc_long_infer = candidate_df_acc_long_infer.merge(gdf, on=['article_id', 'family'], how='left')
    print(count_df)

    # print(candidate_df_acc_long_infer)
    candidate_df_acc_normal = candidate_df_acc[~candidate_df_acc["article_id"].isin(long_table_articles)].reset_index(drop=True)
    candidate_df_nornal_infer = pd.concat([candidate_df_doi, candidate_df_acc_normal]).reset_index(drop=True)

    # add context
    if cfg.use_v2_context:
        candidate_df_nornal_infer["context"] = candidate_df_nornal_infer.parallel_apply(lambda row: get_context_v2(cfg, row, chunk_tokenizer), axis=1)
    else:
        candidate_df_nornal_infer["context"] = candidate_df_nornal_infer.parallel_apply(lambda row: get_context(cfg, row, chunk_tokenizer), axis=1)

    if len(candidate_df_acc_long_infer) > 0:
        candidate_df_acc_long_infer["context"] = candidate_df_acc_long_infer.parallel_apply(lambda row: get_context_many_accessions(cfg, row, chunk_tokenizer), axis=1)
        candidate_df = pd.concat([candidate_df_nornal_infer, candidate_df_acc_long_infer]).reset_index(drop=True)
    else:
        candidate_df = candidate_df_nornal_infer.copy()
    
    ##########################################
    # candidate_df["context"] = candidate_df.parallel_apply(lambda row: get_context(cfg, row, chunk_tokenizer), axis=1)

    candidate_df = candidate_df.drop(columns=["text"])
    print(f"shape of candidate_df: {candidate_df.shape}")

    # Load LLM
    max_model_len = cfg.max_model_len
    model_path = cfg.model_name

    llm = vllm.LLM(
        model_path,
        tensor_parallel_size=torch.cuda.device_count(),
        quantization=cfg.quantization,
        gpu_memory_utilization=0.95,
        trust_remote_code=True,
        dtype="half",
        enforce_eager=True,
        max_model_len=max_model_len,
        disable_log_stats=True,
        enable_prefix_caching=True,
        cpu_offload_gb=cfg.cpu_offload_gb,
        max_num_seqs=cfg.max_num_seqs,
        swap_space=cfg.swap_space,
        max_num_batched_tokens=max_model_len,
    )

    tokenizer = llm.get_tokenizer()

    # create prompts ---
    prompts = []

    for i, row in candidate_df.iterrows():
        article_id = row["article_id"]
        dataset_id = row["dataset_id"]
        context = row["context"]

        user_prompt = USER_TEMPLATE.format(context=context, dataset_id=dataset_id)
        messages = [{"role": "system", "content": SYS_PROMPT}, {"role": "user", "content": user_prompt}]

        # ---
        prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

        tokens = tokenizer.encode(prompt)
        token_length = len(tokens)
        extra_tokens = token_length - (max_model_len - 16)
        if extra_tokens > 0:
            prompt = tokenizer.decode(tokens[extra_tokens:])

        prompts.append(prompt)

    # show
    nprompt = min(2, len(prompts))
    show_prompts = random.sample(prompts, nprompt)
    for i, prompt in enumerate(show_prompts):
        print(f"Prompt {i}: {prompt}")
        print("-" * 100)

    # vllm
    outputs = llm.generate(prompts, vllm.SamplingParams(temperature=0.0, skip_special_tokens=True, max_tokens=1, logprobs=20), use_tqdm=True)

    probs_list = []
    for output in outputs:
        lps = output.outputs[0].logprobs[0]
        probs_list.append(get_probs(lps))

    sub_df = pd.DataFrame()
    sub_df["article_id"] = candidate_df["article_id"]
    sub_df["dataset_id"] = candidate_df["dataset_id"]
    sub_df["probs"] = probs_list

    sub_df = sub_df.groupby(["article_id", "dataset_id"])["probs"].apply(lambda x: np.mean(x, axis=0)).reset_index()
    choices = ["Primary", "Secondary", "NA"]
    sub_df["type"] = sub_df["probs"].apply(lambda x: choices[np.argmax(x)])

    # save --
    sub_df.to_parquet(os.path.join(cfg.save_dir, cfg.output_file))

Writing classify_candidates.py


In [18]:
%%writefile config_classify_14b.yaml

save_dir: ./working
output_file: preds_14b.parquet
candidate_path: ./working/candidates_infer.parquet

model_name: /kaggle/input/mdc_qwen14b_merged/transformers/default/14 # /kaggle/input/mdc_qwen14b_merged/transformers/default/9
max_model_len: 3600
cpu_offload_gb: 2
max_num_seqs: 1
quantization:
swap_space: 0

context:
  char_per_chunk: 1400
  char_overlap: 100
  top_k: 2
  max_hits: 5
  relevance_th: 8.0
  max_doi_chunk: 1
  token_budget: 3072

xml_first: false
use_v2_context: true # false
max_accession_ids_per_family: 8

Writing config_classify_14b.yaml


In [19]:
%%time
!python classify_candidates.py --config_path config_classify_14b.yaml

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Cache directory (cache/exp_tmp/txt) exists? -> True
# of previously processed files: 524
# of new files processed: 0
shape of candidates: (54, 2)
shape before search for dataset_id in text: (108, 4)
shape after search for dataset_id in text: (93, 4)
shape after deduplication: (54, 4)
                 article_id family  count
0  10.1021_acsomega.3c06074    pdb      1
   0.00%                                          |        0 /       14 |                          
   0.00%                                          |        0 /       14 |                          
   0.00%                                          |        0 /       13 |                          
   0.00%                                          |        0 /       13 |                          

# 32B Model

In [20]:
%%writefile prep_difficult.py

import argparse
import pandas as pd

def get_uncertain_samples(oof_df, frac=0.2):
    oof_df = oof_df.copy()

    oof_df["null_p"] = oof_df["probs"].apply(lambda x: x[2])
    oof_df = oof_df[oof_df["null_p"] < 0.9].reset_index(drop=True)
    oof_df["diff"] = oof_df["probs"].apply(lambda x: abs(x[1] - x[0]))

    n_samples = int(len(oof_df) * frac)
    n_samples = max(1, n_samples)
    
    uncertain_df = oof_df.sort_values(by="diff").head(n_samples).reset_index(drop=True)
    uncertain_df = uncertain_df.drop(columns=["diff", "null_p"])

    return uncertain_df


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--input_path", type=str)
    ap.add_argument("--output_path", type=str)
    ap.add_argument("--frac_doi", type=float, default=0.5)
    ap.add_argument("--frac_acc", type=float, default=0.1)

    args = ap.parse_args()

    pred_df = pd.read_parquet(args.input_path)
    print(f"shape of input: {pred_df.shape}")
    pred_df_doi = pred_df[pred_df['dataset_id'].str.startswith("https://doi.org/")].reset_index(drop=True)
    pred_df_acc = pred_df[~pred_df['dataset_id'].str.startswith("https://doi.org/")].reset_index(drop=True)

    uncertain_df_doi = get_uncertain_samples(pred_df_doi, args.frac_doi)
    uncertain_df_acc = get_uncertain_samples(pred_df_acc, args.frac_acc)
    
    uncertain_df = pd.concat([uncertain_df_doi, uncertain_df_acc]).reset_index(drop=True)
    
    uncertain_df.to_parquet(args.output_path)
    print(f"shape of output: {uncertain_df.shape}")

Writing prep_difficult.py


In [21]:
!python prep_difficult.py --input_path ./working/preds_14b.parquet --output_path ./working/infer_uncertain_32b.parquet --frac_doi 0.5 --frac_acc 0.2

shape of input: (54, 4)
shape of output: (27, 4)


In [22]:
# df = pd.read_parquet("./working/infer_uncertain_32b.parquet")
# df.groupby(['article_id', 'dataset_id'])['type'].agg('count').reset_index()

In [23]:
%%writefile config_classify_32b.yaml

save_dir: ./working
output_file: preds_32b.parquet
candidate_path: ./working/infer_uncertain_32b.parquet

model_name: /kaggle/input/mdc_cls_qwen_32b_awq/transformers/default/2
max_model_len: 3600
cpu_offload_gb: 2
max_num_seqs: 1
quantization: awq
swap_space: 0

context:
  char_per_chunk: 1400
  char_overlap: 100
  top_k: 2
  max_hits: 5
  relevance_th: 8.0
  max_doi_chunk: 1
  token_budget: 3072

xml_first: true
use_v2_context: false
max_accession_ids_per_family: 8

Writing config_classify_32b.yaml


In [24]:
%%time
!python classify_candidates.py --config_path config_classify_32b.yaml

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Cache directory (cache/exp_tmp/txt) exists? -> True
# of previously processed files: 524
# of new files processed: 0
shape of candidates: (27, 2)
shape before search for dataset_id in text: (54, 4)
shape after search for dataset_id in text: (45, 4)
shape after deduplication: (27, 4)
                 article_id family  count
0  10.1021_acsomega.3c06074    pdb      1
   0.00%                                          |        0 /        7 |                          
   0.00%                                          |        0 /        7 |                          
   0.00%                                          |        0 /        7 |                          
   0.00%                                          |        0 /        6 |                          

# 72b Model

In [25]:
!python prep_difficult.py --input_path ./working/preds_32b.parquet --output_path ./working/infer_uncertain_72b.parquet --frac_doi 0.1 --frac_acc 0.1

shape of input: (27, 4)
shape of output: (3, 4)


In [26]:
%%writefile config_classify_72b.yaml

save_dir: ./working
output_file: preds_72b.parquet
candidate_path: ./working/infer_uncertain_72b.parquet

model_name: /kaggle/input/mdc_cls_qwen_72b_awq/transformers/default/2 # /kaggle/input/mdc_cls_qwen_72b_awq/transformers/default/1
max_model_len: 2560
cpu_offload_gb: 8
max_num_seqs: 1
quantization: awq
swap_space: 1

context:
  char_per_chunk: 1200
  char_overlap: 100
  top_k: 0
  max_hits: 3
  relevance_th: 5.0
  max_doi_chunk: 1
  token_budget: 2048

xml_first: false
use_v2_context: false
max_accession_ids_per_family: 8

Writing config_classify_72b.yaml


In [27]:
%%time
!python classify_candidates.py --config_path config_classify_72b.yaml

INFO: Pandarallel will run on 4 workers.
INFO: Pandarallel will use Memory file system to transfer data between the main process and workers.
Cache directory (cache/exp_tmp/txt) exists? -> True
# of previously processed files: 524
# of new files processed: 0
shape of candidates: (3, 2)
shape before search for dataset_id in text: (6, 4)
shape after search for dataset_id in text: (6, 4)
shape after deduplication: (3, 4)
                 article_id family  count
0  10.1021_acsomega.3c06074    pdb      1
   0.00%                                          |        0 /        1 |                          
   0.00%                                          |        0 /        1 |                          
   0.00%                                          |        0 /        1 |                          MM   0.00%                                          |        0 /        1 |                   

# Prepare Sub

In [28]:
import os
import re
import numpy as np
import pandas as pd

In [29]:
pred_14b = pd.read_parquet("./working/preds_14b.parquet")
pred_32b = pd.read_parquet("./working/preds_32b.parquet")
pred_72b = pd.read_parquet("./working/preds_72b.parquet")

pred_assumed = pd.read_parquet("./working/acc_preds_assumed.parquet")

sub_df = pd.concat([pred_72b, pred_32b, pred_32b, pred_14b, pred_assumed]).reset_index(drop=True)
# sub_df = pd.concat([pred_32b, pred_32b, pred_14b, pred_assumed]).reset_index(drop=True)

sub_df = sub_df.groupby(["article_id", "dataset_id"])["probs"].apply(lambda x: np.mean(x, axis=0)).reset_index()

sub_df.shape

(64, 3)

In [30]:
# pred_72b

In [31]:
base_df = pd.read_parquet("./working/base_pred_df.parquet")
family_df = pd.merge(base_df, sub_df, on=['article_id', 'dataset_id'], how='inner')
family_df = family_df.groupby(['article_id', 'family'])["probs"].apply(lambda x: np.mean(x, axis=0)).reset_index().rename(columns={'probs': 'family_probs'})
family_df.sample()

,article_id,family,family_probs
12,10.1002_esp.5058,doi,"[0.8585518983017569, 0.008752240721316846, 0.1..."


In [32]:
base_df = pd.read_parquet("./working/base_pred_df.parquet")
oof_df = pd.merge(base_df, sub_df, on=['article_id', 'dataset_id'], how='left')
oof_df = oof_df.merge(family_df, on=['article_id', 'family'])

def fill_probs(row):
    try:
        return list(row['probs'])
    except Exception:
        return list(row['family_probs'])
    
oof_df['probs'] = oof_df.apply(lambda row: fill_probs(row), axis=1)
oof_df = oof_df.drop(columns=['family', 'family_probs']).reset_index(drop=True)
oof_df.shape

(64, 3)

In [33]:
oof_df

,article_id,dataset_id,probs
0,10.1002_2017jc013030,https://doi.org/10.17882/47142,"[0.27898905429144644, 0.5117241302197556, 0.20..."
1,10.1002_2017jc013030,https://doi.org/10.17882/49388,"[0.260569768340067, 0.5130554702563, 0.2263747..."
2,10.1002_cssc.202201821,https://doi.org/10.5281/zenodo.7074790,"[0.3217902428813044, 0.40419938118034, 0.27401..."
3,10.1002_ece3.3985,https://doi.org/10.4159/harvard.9780674433960,"[0.006859003104898642, 0.006493973618253573, 0..."
4,10.1002_ece3.4466,https://doi.org/10.5061/dryad.r6nq870,"[0.846957838615053, 4.888142627977436e-09, 0.1..."
...,...,...,...
59,10.1021_acsomega.3c06074,CHEMBL3422978,"[0.0, 1.0, 0.0]"
60,10.1021_acsomega.3c06074,CHEMBL572163,"[0.0, 1.0, 0.0]"
61,10.1021_acsomega.3c06074,CHEMBL1782574,"[0.0, 1.0, 0.0]"
62,10.1021_acsomega.3c06074,CHEMBL390649,"[0.0, 1.0, 0.0]"


### Accession Removal based on sum of primary probs

In [34]:
import pandas as pd

def should_remove_accession_id(row, buffer, th=0.5):
    article_id = row["article_id"]
    dataset_id = row["dataset_id"]
    max_doi_prob = buffer.get(article_id, 0.0)

    if dataset_id.startswith("https://doi.org/"):
        return False

    if max_doi_prob > th:
        return True

    return False


# ----
buffer = dict()
for _, row in oof_df.iterrows():
    article_id = row["article_id"]
    dataset_id = row["dataset_id"]
    if dataset_id.startswith("https://doi.org/"):
        p = round(float(row["probs"][0]), 4)
        if article_id in buffer:
            buffer[article_id] += p
        else:
            buffer[article_id] = p

In [35]:
print(f"shape before: {oof_df.shape}")
oof_df["should_remove"] = oof_df.apply(lambda row: should_remove_accession_id(row, buffer, th=0.8), axis=1)
oof_df = oof_df[~oof_df["should_remove"]].reset_index(drop=True)
oof_df = oof_df.drop(columns=["should_remove"])
print(f"shape after: {oof_df.shape}")

shape before: (64, 3)
shape after: (64, 3)


### Filter NA

In [36]:
def type_filter_fn(row, doi_th, acc_th):
    is_doi = row["dataset_id"].startswith("https://doi.org/")
    
    if is_doi:
        return row["null_p"] <= doi_th
    else:
        return row["null_p"] <= acc_th

In [37]:
oof_df['null_p'] = oof_df['probs'].apply(lambda x: x[2])

doi_th = 0.6
acc_th = 0.9

oof_df["should_keep"] = oof_df.apply(lambda x: type_filter_fn(x, doi_th, acc_th), axis=1)
oof_df = oof_df[oof_df["should_keep"]].reset_index(drop=True)
oof_df = oof_df.drop(columns=["should_keep"])
oof_df = oof_df.reset_index(drop=True)

oof_df['type'] = oof_df['probs'].apply(lambda x: 'Primary' if x[0] > x[1] else 'Secondary')
oof_df['type'].value_counts()

type
Secondary    37
Primary      26
Name: count, dtype: int64

In [38]:
oof_df.shape

(63, 5)

# Submission

In [39]:
oof_df['row_id'] = range(len(oof_df))
oof_df = oof_df[oof_df["type"].isin(["Primary", "Secondary"])].reset_index(drop=True)
oof_df = oof_df.drop_duplicates(subset=['article_id', 'dataset_id'], keep="first").reset_index(drop=True)
oof_df = oof_df[['row_id', 'article_id', 'dataset_id', 'type']].copy()

oof_df

,row_id,article_id,dataset_id,type
0,0,10.1002_2017jc013030,https://doi.org/10.17882/47142,Secondary
1,1,10.1002_2017jc013030,https://doi.org/10.17882/49388,Secondary
2,2,10.1002_cssc.202201821,https://doi.org/10.5281/zenodo.7074790,Secondary
3,3,10.1002_ece3.4466,https://doi.org/10.5061/dryad.r6nq870,Primary
4,4,10.1002_ece3.5260,https://doi.org/10.5061/dryad.2f62927,Primary
...,...,...,...,...
58,58,10.1021_acsomega.3c06074,CHEMBL3422978,Secondary
59,59,10.1021_acsomega.3c06074,CHEMBL572163,Secondary
60,60,10.1021_acsomega.3c06074,CHEMBL1782574,Secondary
61,61,10.1021_acsomega.3c06074,CHEMBL390649,Secondary


In [40]:
oof_df.to_csv("submission.csv", index=False, columns=["row_id", "article_id", "dataset_id", "type"])

In [41]:
print("Final submission stats:")
print(oof_df["type"].value_counts())
print(f"Total entries: {len(oof_df)}")

Final submission stats:
type
Secondary    37
Primary      26
Name: count, dtype: int64
Total entries: 63


# Evaluate validation score

In [42]:
# Validation scoring
import os

def f1_score(tp, fp, fn):
    return 2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) != 0 else 0.0
    
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    pred_df = pd.read_csv("submission.csv")

    label_df = pd.read_csv("/kaggle/input/make-data-count-finding-data-references/train_labels.csv")
    blacklist = set(label_df[label_df["type"] == "Missing"]["article_id"].unique().tolist())
    label_df = label_df[label_df['type'] != 'Missing'].reset_index(drop=True)

    pred_df = pred_df[~pred_df['article_id'].isin(blacklist)].reset_index(drop=True)

    hits_df = label_df.merge(pred_df, on=["article_id", "dataset_id", "type"])
    
    tp = hits_df.shape[0]
    fp = pred_df.shape[0] - tp
    fn = label_df.shape[0] - tp
    
    print("\nValidation Results:")
    print("TP:", tp)
    print("FP:", fp)
    print("FN:", fn)
    print("F1 Score:", round(f1_score(tp, fp, fn), 3))


Validation Results:
TP: 47
FP: 12
FN: 672
F1 Score: 0.121
